# Phase 2 SFT Warm-Start — Qwen3-1.7B on Kaggle T4×2

**Pre-onsite mode.** Substitutes `Qwen/Qwen3-1.7B` for the on-site default `Qwen/Qwen3-4B` (which is too tight on T4×2 with GRPO rollouts in Phase 3).

**Pipeline position:** Phase 1 (CPU oracle trajectories, run on laptop) → **Phase 2 (THIS notebook, T4×2, ~2-3h)** → Phase 3 (GRPO polish, separate Kaggle notebook).

**Before running:**
1. Add Kaggle Dataset containing `sft_trajectories.jsonl` (uploaded from `envs/reconcile_gst2b_env/data/sft_trajectories.jsonl` on laptop after Phase 1 completes). Update `DATASET_SLUG` in cell 4.
2. Enable accelerator: `Settings → Accelerator → GPU T4 x2`.
3. Enable internet: `Settings → Internet → On` (needed for HF model download).

**Pass criterion:** train loss < 0.8, no OOM, `data/sft_checkpoint/final/adapter_model.safetensors` present.

**Output:** LoRA adapter at `/kaggle/working/sft_checkpoint/final/`. Download as zip and feed into Phase 3 GRPO notebook.

## Cell 1 — environment check

In [ ]:
!nvidia-smi
!python --version
import torch

print("cuda:", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(
        f"  [{i}] {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB"
    )

## Cell 2 — clone repo

In [ ]:
import os

os.chdir("/kaggle/working")

# Replace with your fork's HTTPS URL. Branch must be scaffold/reconcile-gst2b.
REPO_URL = "https://github.com/akashkathole7/OpenEnv.git"
BRANCH = "scaffold/reconcile-gst2b"

!rm -rf /kaggle/working/OpenEnv
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /kaggle/working/OpenEnv
os.chdir("/kaggle/working/OpenEnv")
!git log --oneline -3
!ls envs/reconcile_gst2b_env/scripts/ | head -20

## Cell 3 — install training deps

Pinned to TRL >= 0.21 (script asserts this). PEFT + accelerate + bitsandbytes for LoRA. `torchao>=0.16` clears newer transformers' version floor.

**Important:** the `--upgrade` calls can clobber Kaggle's CUDA torch with a CPU-only wheel (observed: `2.10.0+cu128 -> 2.10.0+cpu` after transformers upgrade). The final `--force-reinstall torch --index-url https://download.pytorch.org/whl/cu128` step restores the CUDA wheel; the `torch.cuda.is_available()` assertion fails fast if it didn't stick.


In [ ]:
!pip install -q --upgrade 'trl>=0.21,<1.0' peft accelerate bitsandbytes datasets 'torchao>=0.16.0'
!pip install -q --upgrade transformers
!pip install -q openenv-core 2>/dev/null || echo 'openenv-core not on PyPI — using src/ from clone'

# CRITICAL: the --upgrade calls above sometimes resolve torch to a CPU-only
# wheel on Kaggle Python 3.12 (observed: 2.10.0+cu128 -> 2.10.0+cpu after
# transformers upgrade). Force-reinstall the CUDA wheel last so training
# runs on T4, not CPU.
!pip install -q --force-reinstall torch --index-url https://download.pytorch.org/whl/cu128

# Verify versions + CUDA availability. Fail fast here rather than discover
# at model-load time.
import torch, trl, peft, transformers
try:
    import torchao
    ao_ver = torchao.__version__
except ImportError:
    ao_ver = '<not installed>'
print(f'torch: {torch.__version__}  | cuda: {torch.cuda.is_available()}')
print(f'trl: {trl.__version__} | peft: {peft.__version__} | transformers: {transformers.__version__} | torchao: {ao_ver}')
assert torch.cuda.is_available(), (
    'CUDA torch not available. Check --index-url pulled a +cu128 wheel '
    'and Kaggle Accelerator is GPU T4 x2.'
)
# Also assert TRL is in the pre-1.x stable range (1.x SFTTrainer observed
# to hang silently on Kaggle T4 with our config).
assert trl.__version__.split('.')[0] == '0', (
    f'Need TRL 0.21.x (stable SFTTrainer). Got {trl.__version__}. '
    'Check the pip install pin: trl>=0.21,<1.0'
)
print('deps OK. Proceed to cell 8.')

## Cell 4 — mount sft_trajectories.jsonl from Kaggle Dataset

Replace `DATASET_SLUG` with the actual Kaggle Dataset slug after upload (e.g. `akashkathole/sft-trajectories-gst2b`). The dataset should contain a single file `sft_trajectories.jsonl`.

In [ ]:
import os, glob, json

# Expected: sft_trajectories.jsonl, 3000 rows, ~59 MB (balanced subsample:
#   1000 oracle + 1000 inspect_then_label + 1000 supplier_cap_aware,
#   round-robin interleaved so any prefix preserves the policy balance).
DATASET_SLUG = "akashkathole/sft-trajectories-gst2b"  # <-- UPDATE ME

# Self-diagnostic: walk every file under /kaggle/input/ so we ship the full
# mount layout in the cell output (Kaggle uses at least two mount
# conventions: /kaggle/input/<slug>/ and /kaggle/input/datasets/<user>/<slug>/).
print("=== /kaggle/input/ tree (top 60 files) ===")
count = 0
for root, dirs, files in os.walk("/kaggle/input"):
    for f in sorted(files):
        fp = os.path.join(root, f)
        try:
            size = os.path.getsize(fp) / 1e6
            print(f"  {fp}  ({size:.2f} MB)")
        except OSError:
            print(f"  {fp}  (size unavailable)")
        count += 1
        if count >= 60:
            break
    if count >= 60:
        break
print()

# Search /kaggle/input/** broadly so this works regardless of Kaggle mount
# convention (both /kaggle/input/<slug>/ and /kaggle/input/datasets/<user>/<slug>/
# are seen in practice). Three filename patterns, first non-empty match wins.
patterns = [
    "/kaggle/input/**/sft_trajectories*.jsonl",
    "/kaggle/input/**/sft-trajectories*.jsonl",
    "/kaggle/input/**/*.jsonl",
]
candidates = []
matched_pattern = None
for p in patterns:
    hits = sorted(glob.glob(p, recursive=True))
    if hits:
        candidates = hits
        matched_pattern = p
        break

assert candidates, (
    "No *.jsonl found under /kaggle/input/ via any of:\n"
    + "\n".join(f"  {p}" for p in patterns)
    + "\nSee the filesystem walk above for what is actually mounted."
)

JSONL_SRC = candidates[0]
src_size_mb = os.path.getsize(JSONL_SRC) / 1e6
print(f"matched via: {matched_pattern}")
print(f"found:       {JSONL_SRC} ({src_size_mb:.1f} MB)")
assert 50 < src_size_mb < 70, (
    f"Unexpected size {src_size_mb:.1f} MB - expected ~55-65 MB for "
    f"3000-row balanced subsample. Re-check dataset upload."
)

# Symlink into the path train_sft_warmstart expects.
DATA_DIR = "/kaggle/working/OpenEnv/envs/reconcile_gst2b_env/data"
os.makedirs(DATA_DIR, exist_ok=True)
DST = f"{DATA_DIR}/sft_trajectories.jsonl"
if os.path.lexists(DST):
    os.remove(DST)
os.symlink(JSONL_SRC, DST)
print(f"symlinked:   {DST} -> {os.readlink(DST)}")
print()

# Sanity: count rows + verify balance + show first row's keys.
with open(DST) as f:
    rows = [json.loads(line) for line in f]
assert len(rows) == 3000, (
    f"Expected exactly 3000 rows, got {len(rows)}. Re-check dataset upload."
)
mean_total = sum(r["total"] for r in rows) / len(rows)
print(f"rows:        {len(rows)} | mean total: {mean_total:.3f}")
print(f"first row:   keys={list(rows[0].keys())}")

## Cell 5 — dry-run smoke (CPU, ~30s)

Verifies the script loads, tokenizer patches, dataset parses, LoRA attaches, SFTTrainer accepts config. **Do this before the 2-3h real run** so we catch config errors in seconds, not hours.

In [ ]:
!cd /kaggle/working/OpenEnv && CUDA_VISIBLE_DEVICES=0 PYTHONPATH=src:envs python -m \
    envs.reconcile_gst2b_env.scripts.train_sft_warmstart \
    --input-jsonl envs/reconcile_gst2b_env/data/sft_trajectories.jsonl \
    --model Qwen/Qwen3-1.7B \
    --dry-run 2>&1 | tail -40

## Cell 6 — real Phase 2 SFT run (single T4, ~2-3h)

**Mode-substituted from briefing on-site command:** `Qwen/Qwen3-1.7B` instead of `Qwen/Qwen3-4B` (fits a single T4 with GRPO rollout headroom for Phase 3).

**`CUDA_VISIBLE_DEVICES=0` forces single-GPU** to avoid pipeline-parallel NCCL deadlock on T4×2 (observed: 5h+ silent hang when model was split across 2 T4s).

**fp16 (not bf16) on T4.** Turing tensor cores are fp16-only; bf16 falls back to slow CUDA-core compute (~235s/step observed). With fp16 and max_seq_length=2048, per-step drops to ~30-50s and total wall time fits in Kaggle's 9h session.

Batch config: `--batch-size 1 --grad-accum 8` per briefing. If OOM, drop `grad-accum` to 4.

Log is teed to `/kaggle/working/sft_run.log` so it survives kernel timeouts.

In [ ]:
!cd /kaggle/working/OpenEnv && CUDA_VISIBLE_DEVICES=0 PYTHONPATH=src:envs python -m \
    envs.reconcile_gst2b_env.scripts.train_sft_warmstart \
    --input-jsonl envs/reconcile_gst2b_env/data/sft_trajectories.jsonl \
    --output-dir  /kaggle/working/sft_checkpoint \
    --model       Qwen/Qwen3-1.7B \
    --epochs      1 \
    --batch-size  1 \
    --grad-accum  8 \
    --learning-rate 2e-5 \
    --precision   fp16 \
    --max-seq-length 2048 \
    --logging-steps 1 \
    --save-steps   25 \
    2>&1 | tee /kaggle/working/sft_run.log

## Cell 7 — verify checkpoint + zip for download

**Pass gates:**
- `adapter_model.safetensors` exists in `final/`
- `sft_summary.json` shows final loss < 0.8
- Total adapter size ~50-150 MB (LoRA rank 16 on 1.7B base)

In [ ]:
import os, json, glob

CKPT_DIR = "/kaggle/working/sft_checkpoint/final"
assert os.path.isdir(CKPT_DIR), (
    f"No final checkpoint at {CKPT_DIR} — training may have crashed. Check /kaggle/working/sft_run.log."
)

print("=== checkpoint contents ===")
for f in sorted(os.listdir(CKPT_DIR)):
    sz = os.path.getsize(os.path.join(CKPT_DIR, f)) / 1e6
    print(f"  {f}  ({sz:.2f} MB)")

summary_path = "/kaggle/working/sft_checkpoint/sft_summary.json"
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print("\n=== sft_summary.json ===")
    print(json.dumps(summary, indent=2))
    train_loss = summary.get("metrics", {}).get("train_loss")
    if train_loss is not None:
        verdict = (
            "PASS"
            if train_loss < 0.8
            else (
                "UNDERFIT"
                if train_loss > 1.5
                else "OVERFIT"
                if train_loss < 0.2
                else "PASS"
            )
        )
        print(
            f"\nfinal train loss: {train_loss:.4f}  ({verdict} per briefing thresholds)"
        )

# Zip for easy download.
!cd /kaggle/working && zip -r sft_checkpoint.zip sft_checkpoint/ -x '*.bin' > /dev/null
!ls -lh /kaggle/working/sft_checkpoint.zip
print(
    "\n→ download sft_checkpoint.zip from the Output panel and feed into the Phase 3 GRPO notebook."
)

## What to do next

1. Download `sft_checkpoint.zip` from Kaggle Output.
2. Upload it as a new Kaggle Dataset (e.g. `akashkathole/sft-checkpoint-gst2b-1p7b`).
3. Open the Phase 3 GRPO notebook (TBD), point `--model` at the SFT checkpoint dataset path.

**If something failed:**
- CUDA OOM on model load → drop `--grad-accum` from 8 to 4 (decision tree, Phase 2 failures table).
- TRL version error → re-run cell 3, the `pip install -q --upgrade 'trl>=0.21'` should fix it.
- Final loss > 1.5 (under-fit) → increase `--epochs` to 2, check Phase 1 row count.
- Final loss < 0.2 (over-fit) → shuffle JSONL, trim to 500 rows, rerun.